# MovieLens

## Descripción de MovieLens

Este conjunto de datos (ml-32m) describe la actividad de calificación de películas mediante una escala de 5 estrellas y la actividad de etiquetado mediante texto libre de MovieLens, un servicio de recomendación de películas. Contiene 32.000.204 calificaciones y 2.000.072 aplicaciones de etiquetas correspondientes a 87.585 películas. Estos datos fueron generados por 200.948 usuarios entre el 9 de enero de 1995 y el 12 de octubre de 2023. Este conjunto de datos fue generado el 13 de octubre de 2023.

Los usuarios fueron seleccionados aleatoriamente para su inclusión. Todos los usuarios seleccionados habían calificado al menos 20 películas. No se incluye información demográfica. Cada usuario está representado por un identificador (ID) y no se proporciona ninguna otra información.

**Formato y codificación**

Los archivos del conjunto de datos están escritos como archivos de valores separados por comas (CSV), con una única fila de encabezado. Las columnas que contienen comas (,) se escapan utilizando comillas dobles (").

**Identificadores de usuario**

Los usuarios de MovieLens fueron seleccionados aleatoriamente para su inclusión. Sus identificadores han sido anonimizados.

Los identificadores de usuario son consistentes entre ratings.csv y tags.csv; es decir, el mismo ID hace referencia al mismo usuario en ambos archivos.

**Identificadores de películas**

Solo se incluyen en el conjunto de datos las películas que tienen al menos una calificación o una etiqueta.

Los identificadores de las películas son consistentes entre ratings.csv, tags.csv, movies.csv y links.csv; es decir, el mismo ID hace referencia a la misma película en los cuatro archivos de datos.

**Estructura del archivo de datos de calificaciones (ratings.csv)**

Todas las calificaciones están contenidas en el archivo ratings.csv. Cada línea de este archivo después de la fila de encabezado representa la calificación de una película por parte de un usuario y tiene el siguiente formato:

userId,movieId,rating,timestamp

Las líneas de este archivo están ordenadas primero por userId y, dentro de cada usuario, por movieId.

Las calificaciones se realizan en una escala de 5 estrellas, con incrementos de media estrella (0,5 estrellas a 5,0 estrellas).

Los valores de timestamp representan los segundos transcurridos desde la medianoche del Tiempo Universal Coordinado (UTC) del 1 de enero de 1970.

**Estructura del archivo de datos de etiquetas (tags.csv)**

Todas las etiquetas están contenidas en el archivo tags.csv. Cada línea de este archivo después de la fila de encabezado representa una etiqueta aplicada a una película por un usuario y tiene el siguiente formato:

userId,movieId,tag,timestamp

Las líneas de este archivo están ordenadas primero por userId y, dentro de cada usuario, por movieId.

Las etiquetas son metadatos generados por los usuarios sobre las películas. Cada etiqueta suele ser una sola palabra o una frase corta. El significado, valor y propósito de una etiqueta determinada son definidos por cada usuario.

Los valores de timestamp representan los segundos transcurridos desde la medianoche del Tiempo Universal Coordinado (UTC) del 1 de enero de 1970.

**Estructura del archivo de datos de películas (movies.csv)**

La información de las películas está contenida en el archivo movies.csv. Cada línea de este archivo después de la fila de encabezado representa una película y tiene el siguiente formato:

movieId,title,genres

Los títulos de las películas se introducen manualmente o se importan desde https://www.themoviedb.org/
 y contienen el año de estreno entre paréntesis. Pueden existir errores e inconsistencias en estos títulos.

Los géneros están separados mediante el carácter de barra vertical (|) y se seleccionan de la siguiente lista:

* Action — Acción
* Adventure — Aventura
* Animation — Animación
* Children's — Infantil
* Comedy — Comedia
* Crime — Crimen
* Documentary — Documental
* Drama — Drama
* Fantasy — Fantasía
* Film-Noir — Cine negro
* Horror — Terror
* Musical — Musical
* Mystery — Misterio
* Romance — Romance
* Sci-Fi — Ciencia ficción
* Thriller — Suspense
* War — Bélico
* Western — Western
* (no genres listed) — (sin géneros indicados)

## Importación de librerías y datos

In [2]:
import pandas as pd

In [3]:
movies = pd.read_csv('../../data/raw/movies.csv')
tags = pd.read_csv('../../data/raw/tags.csv')
ratings = pd.read_csv('../../data/raw/ratings.csv')

In [4]:
movies_raw = movies.copy()
tags_raw = tags.copy()
ratings_raw = ratings.copy()

## Calidad y limpieza de datos

In [5]:
def print_shape_columns(df):
    print(f"Registros: {df.shape[0]}\nColumnas: {df.shape[1]}\n")
    print(f"Columnas: {df.columns}\n")

def print_missing_values(df):
    print(f"Porcentaje de valores faltantes: {df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100:.2f}%\n")
    print("Porcentaje de valores faltantes por columna:")
    display(df.isnull().sum() / df.shape[0] * 100)

### Movies

In [6]:
# Vistazo a los datos
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
print_shape_columns(movies)

Registros: 87585
Columnas: 3

Columnas: Index(['movieId', 'title', 'genres'], dtype='str')



Ya que el año de lanzamiento de la película está contenida dentro de `title`, se crea una nueva columna `year` que contenga ese año para cada película

In [8]:
# Se eliminan los espacios en blanco al inicio y al final de los títulos de las películas
movies['title'] = movies['title'].str.strip()

prop_no_year = movies[~movies['title'].str.contains(r'\(\d{4}\)$')].shape[0] / movies.shape[0] * 100
print(f"Porcentaje de películas sin año en el título: {prop_no_year:.3f}%")

movies['release_year'] = movies['title'].str.extract(r'\((\d{4})\)$')
movies['release_year'] = pd.to_datetime(movies['release_year'], format='%Y', errors='coerce')
movies['title'] = movies['title'].str.replace(r'\s*\(\d{4}\)$', '', regex=True)

Porcentaje de películas sin año en el título: 0.704%


Se eliminan las películas sin año de lanzamiento

In [9]:
movies = movies[movies['release_year'].notnull()]

De acuerdo al `README.md` del conjunto de datos original, los géneros a los que pertenece una película están listados dentro de `genres`, y separados por '|'. Se crea una lista con los posibles géneros indicados en la descripción de los datos, y posteriormente se crea una columna para cada uno de los géneros, incluyendo una columna `None` para indicar que dicha película no pertenece a ningún género. Estas columnas de género tendrán como únicos valores `0` y `1`, para indicar que la película no pertenece y sí pertenece a dicho género, respectivamente.

In [10]:
valid_genres = ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

movies[valid_genres] = movies['genres'].str.get_dummies(sep='|')[valid_genres]
movies['None'] = (movies[valid_genres].sum(axis=1) == 0).astype(int)

Hay una película que no pertenece a ninguna de los géneros indicados en la descripción de los datos, pero si pertenece al género 'IMAX'. Como solo es una película, de miles, se conservan únicamente los géneros originales, de forma que se trata a esta película como no perteneciente a ninguno de los géneros.

In [11]:
movies[movies['None'] == 1]['genres'].value_counts()

genres
(no genres listed)    6707
IMAX                     1
Name: count, dtype: int64

Se elimina la única película que tiene título vacío ('')

In [12]:
print(f"Cantidad de películas con título vacío: {movies[movies['title'] == ''].shape[0]}")
movies = movies[movies['title'] != '']

Cantidad de películas con título vacío: 1


In [13]:
print_missing_values(movies)

Porcentaje de valores faltantes: 0.00%



Porcentaje de valores faltantes por columna:


movieId         0.0
title           0.0
genres          0.0
release_year    0.0
Action          0.0
Adventure       0.0
Animation       0.0
Children        0.0
Comedy          0.0
Crime           0.0
Documentary     0.0
Drama           0.0
Fantasy         0.0
Film-Noir       0.0
Horror          0.0
Musical         0.0
Mystery         0.0
Romance         0.0
Sci-Fi          0.0
Thriller        0.0
War             0.0
Western         0.0
None            0.0
dtype: float64

Hay parejas de título de película con año de lanzamiento que se repiten en `movies`. Sin embargo, luego de realizar búsquedas en el navegador, estos registros realmente corresponden a películas distintas, a pesar que compartan el mismo título y año de lanzamiento. Ejemplos:
* Aladdin (1992): Una de Walt Disney Pictures, otra de Golden Films / GoodTimes Entertainment.
* Gossip (2000): Una película estadounidense y otra sueca.
* Carmen (2022): Una de Benjamin Millepied, otra de Valerie Buhagiar.

In [14]:
duplicated_title_year = movies[movies.duplicated(subset=['title', 'release_year'], keep=False)].shape[0]
print(f"Cantidad de películas con título y año duplicados: {duplicated_title_year}")

Cantidad de películas con título y año duplicados: 401


Se eliminan las columnas de género, y se mantiene `genre`

In [15]:
movies = movies.drop(columns=valid_genres + ['None'])

### Tags

In [16]:
tags.head()

,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


In [17]:
print_shape_columns(tags)

Registros: 2000072
Columnas: 4

Columnas: Index(['userId', 'movieId', 'tag', 'timestamp'], dtype='str')



Se crea una nueva columna con la fecha y tiempo en que se hizo escribió el `tag` a partir de la columna de marca de tiempo ('timestamp')

In [18]:
tags['tag_datetime'] = pd.to_datetime(tags['timestamp'], unit='s')

# Se elimina 'timestamp'
tags = tags.drop(columns=['timestamp'])

Se eliminan los tags que no hacen referencia a ninguna película

In [19]:
tags = tags[tags['movieId'].isin(movies['movieId'])]

Se eliminan los espacios en blanco al inicio y final de `tag`

In [20]:
tags['tag'] = tags['tag'].str.strip()

Hay 23 tags faltantes en todo `tags`. 15 de esos 23 tags corresponden al mismo usuario (153443) en el mismo día  (2008-01-04). Sin embargo, este usuario sí tiene otros tags válidos en otras fechas, y hubo otros usuarios que también registraron tags válidos en esa fecha. Esto podría indicar un problema particular con ese usuario en esa fecha al momento del registro de datos.

In [21]:
print(f"Cantidad de tags del usuario 153443: {tags[tags['userId'] == 153443].shape[0]}")
print(f"Cantidad de tags registrados el 4 de enero de 2008: {tags[tags['tag_datetime'].dt.date == pd.to_datetime('2008-01-04').date()].shape[0]}")
print(f"Porcentaje de tags con valores faltantes o vacíos: {tags[tags['tag'].isna() | (tags['tag'] == '')].shape[0] / tags.shape[0] * 100:.3f}%")
print("Tags con valores faltantes o vacíos:")
tags[tags['tag'].isna() | (tags['tag'] == '')]

Cantidad de tags del usuario 153443: 66
Cantidad de tags registrados el 4 de enero de 2008: 96
Porcentaje de tags con valores faltantes o vacíos: 0.001%
Tags con valores faltantes o vacíos:


,userId,movieId,tag,tag_datetime
15623,2557,32657,,2009-03-11 20:34:09
185377,27046,33826,NaN,2008-09-15 03:55:08
188558,27411,1265,,2009-03-22 10:18:10
628353,78213,839,,2018-05-27 20:22:26
756418,78213,4247,,2018-05-27 22:40:07
844832,78213,7323,,2018-06-09 02:57:12
1394089,89369,281500,NaN,2022-12-13 14:35:04
1744498,135188,31408,,2008-08-03 20:28:14
1914668,153443,123,NaN,2008-01-04 12:47:47
1914669,153443,346,NaN,2008-01-04 13:05:46


Dado lo anterior, y que la información principal de los tags es el tag en sí, si este valor está faltando, no hay necesidad de mantenerlo, así que se eliminan estos registros.

In [22]:
tags = tags.dropna(subset=['tag'])

Se eliminan los registros con tuplas (`userId`, `movieId`, `tags`) duplicadas

In [23]:
# Se crea una copia para cambiar los tags a minúsculas, sin modificar el df original
tags_copy = tags.copy()
tags_copy['tag'] = tags_copy['tag'].str.lower().str.strip()

tags = tags[tags_copy.duplicated(subset=['userId', 'movieId', 'tag'], keep='first') == False]

### Ratings

In [24]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [25]:
print_shape_columns(ratings)

Registros: 32000204
Columnas: 4

Columnas: Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')



Se crea una nueva columna con la fecha y tiempo en que se hizo la calificación a partir de la columna de marca de tiempo ('timestamp')

In [26]:
ratings['rating_datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')

# Se elimina 'timestamp'
ratings = ratings.drop(columns=['timestamp'])

In [27]:
print_missing_values(ratings)

Porcentaje de valores faltantes: 0.00%

Porcentaje de valores faltantes por columna:


userId             0.0
movieId            0.0
rating             0.0
rating_datetime    0.0
dtype: float64

Se eliminan las calificaciones que no corresponden a ninguna película

In [28]:
ratings = ratings[ratings['movieId'].isin(movies['movieId'])]

In [29]:
print(f"Las posibles calificaciones dadas son las esperadas: {sorted(ratings['rating'].unique().tolist())}")

Las posibles calificaciones dadas son las esperadas: [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]


In [30]:
print(f"Cantidad de usuarios que calificaron varias veces una película: {ratings[ratings.duplicated(subset=['userId', 'movieId'], keep=False)].shape[0]}")

Cantidad de usuarios que calificaron varias veces una película: 0


Se eliminan las calificaciones con fecha previa a la fecha de lanzamiento de la película calificada

In [31]:
movies_ratings = ratings[['movieId', 'rating_datetime']].merge(movies[['movieId', 'release_year']], on='movieId', how='inner')

# Se resetea el índice y se eliminan las calificaciones inválidas
ratings = ratings.reset_index(drop=True)
ratings = ratings[movies_ratings['rating_datetime'] >= movies_ratings['release_year']]

In [32]:
print(f"Cantidad de tags sin calificaciones: {tags[~tags['userId'].isin(ratings['userId'])].shape[0]}")

Cantidad de tags sin calificaciones: 0


## Resumen de datos procesados

### Movies

In [33]:
print_shape_columns(movies)

Registros: 86967
Columnas: 4

Columnas: Index(['movieId', 'title', 'genres', 'release_year'], dtype='str')



In [34]:
print(f"Porcentaje de registros mantenidos: {movies.shape[0]/movies_raw.shape[0] * 100:.3f}%")

Porcentaje de registros mantenidos: 99.294%


In [35]:
print_missing_values(movies)

Porcentaje de valores faltantes: 0.00%



Porcentaje de valores faltantes por columna:


movieId         0.0
title           0.0
genres          0.0
release_year    0.0
dtype: float64

### Tags

In [36]:
print_shape_columns(tags)

Registros: 1992636
Columnas: 4

Columnas: Index(['userId', 'movieId', 'tag', 'tag_datetime'], dtype='str')



In [37]:
print(f"Porcentaje de registros mantenidos: {tags.shape[0]/tags_raw.shape[0] * 100:.3f}%")

Porcentaje de registros mantenidos: 99.628%


In [38]:
print_missing_values(tags)

Porcentaje de valores faltantes: 0.00%

Porcentaje de valores faltantes por columna:


userId          0.0
movieId         0.0
tag             0.0
tag_datetime    0.0
dtype: float64

### Ratings

In [39]:
print_shape_columns(ratings)

Registros: 31963989
Columnas: 4

Columnas: Index(['userId', 'movieId', 'rating', 'rating_datetime'], dtype='str')



In [40]:
print(f"Porcentaje de registros mantenidos: {ratings.shape[0]/ratings_raw.shape[0] * 100:.3f}%")

Porcentaje de registros mantenidos: 99.887%


In [41]:
print_missing_values(ratings)

Porcentaje de valores faltantes: 0.00%

Porcentaje de valores faltantes por columna:


userId             0.0
movieId            0.0
rating             0.0
rating_datetime    0.0
dtype: float64

## Exportación de datos procesados

In [ ]:
# movies_ratings = ratings.merge(movies, on='movieId', how='inner')

In [ ]:
# movies.to_csv('../../data/processed/movies.csv', index=False)
# tags.to_csv('../../data/processed/tags.csv', index=False)
# ratings.to_csv('../../data/processed/ratings.csv', index=False)